In [2]:
import os
import getpass

In [ ]:
import os
os.environ["HF_TOKEN"] = open("hf_env.txt").read().split("=")[1].strip()

In [4]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="novita",
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="meta-llama/Llama-3.2-3B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionOutputMessage(role='assistant', content='The capital of France is Paris.', reasoning=None, tool_call_id=None, tool_calls=None)


In [5]:
def generate_prompt(student):
    """
    Generate a personalized, feature-aware prompt for a learning assistant.
    The tone and recommendations are adapted to the student's class and profile.
    """
    # Decide tone based on educational level
    if student.get("Class_Type") in ["Nursery", "Primary"]:
        tone = "friendly and encouraging"
    elif student.get("Class_Type") in ["Junior Secondary", "Senior Secondary"]:
        tone = "motivational and advisory"
    else:
        tone = "professional and analytical"

    # Collect core student info dynamically
    basic_info = f"""
- Name: {student.get('Name', 'N/A')}
- Age: {student.get('Age', 'N/A')}
- Gender: {student.get('Gender', 'N/A')}
- Class/Level: {student.get('Class_Type', 'N/A')}
- Parent Education: {student.get('Parent_Education', 'N/A')}
- School Type: {student.get('School_Type', 'N/A')}
- Health Status: {student.get('Health', 'N/A')}
- Disability: {student.get('Disability', 'N/A')}
- Socioeconomic Status: {student.get('Socioeconomic_Status', 'N/A')}
"""

    performance_info = f"""
- Overall Performance: {student.get('Performance', 'N/A')}%
- Attendance: {student.get('Attendance', 'N/A')}%
- Dropout Risk: {student.get('Dropout_Risk', 'N/A')}
- Tuition Paid: {student.get('Tuition_Paid', 'N/A')}
- Teacher Comment: {student.get('Teacher_Comment', 'N/A')}
- Test & Quiz Scores: {student.get('Test_Scores', 'N/A')}
- Assignment Completion Rate: {student.get('Assignment_Completion_Rate', 'N/A')}%
- Average Grade/GPA: {student.get('Average_GPA', 'N/A')}%
- Subject Grades: {student.get('Course_Grades', 'N/A')}
- Course-wise Interest: {student.get('Course_Interest_Level', 'N/A')}
- Time Spent per Subject (hrs/week): {student.get('Time_Spent_Per_Subject', 'N/A')}
- Perceived Course Difficulty: {student.get('Course_Difficulty_Perception', 'N/A')}
- Platform Interaction: {student.get('Platform_Login_Frequency', 'N/A')} hrs/week
- Course Interaction Logs: {student.get('Course_Interaction_Logs', 'N/A')}
- Late Submission Rate: {student.get('Late_Submission_Rate', 'N/A')}%
"""

    behavioral_info = f"""
- Learning Style: {student.get('Learning_Style', 'N/A')}
- Attention Span: {student.get('Attention_Span', 'N/A')}
- Preferred Learning Pace: {student.get('Preferred_Learning_Pace', 'N/A')}
- Curiosity Level: {student.get('Curiosity_Level', 'N/A')}
- Procrastination Level: {student.get('Procrastination_Level', 'N/A')}
- Study Consistency: {student.get('Study_Consistency', 'N/A')}
- Focus Quality: {student.get('Focus_Quality', 'N/A')}
- Memory Retention: {student.get('Memory_Retention', 'N/A')}
- Sleep Habits: {student.get('Sleep_Habits', 'N/A')}
- Self-regulation: {student.get('Self_Regulation', 'N/A')}
- Goal-setting Behavior: {student.get('Goal_Setting', 'N/A')}
- Personality: {student.get('Personality', 'N/A')}
- Motivation Type: {student.get('Motivation_Type', 'N/A')}
- Stress Level: {student.get('Stress_Level', 'N/A')}
- Social Interaction Level: {student.get('Social_Interaction_Level', 'N/A')}
- Confidence Level: {student.get('Confidence_Level', 'N/A')}
- Resilience: {student.get('Resilience', 'N/A')}
- Growth Mindset: {student.get('Growth_Mindset', 'N/A')}
- Grit: {student.get('Grit', 'N/A')}
- Anxiety Level: {student.get('Anxiety_Level', 'N/A')}
"""

    system_outputs = f"""
- Course Recommendation System Output: {student.get('Course_Recommendation_System_Output', 'N/A')}
- Inactivity Rate System Output: {student.get('Inactivity_Rate_System_Output', 'N/A')}%
- Failure Rate System Output: {student.get('Failure_Rate_System_Output', 'N/A')}%
"""

    prompt = f"""
You are a highly intelligent personalized learning assistant. 
Use a {tone} tone.

Analyze the student's data below and generate actionable recommendations.

Student Details:
{basic_info}
{performance_info}
{behavioral_info}
{system_outputs}

Please generate:
1. A short feedback paragraph in a {tone} tone.
2. 3-5 specific recommendations for academic improvement and personal growth.
3. 2-3 personalized learning strategies, materials, or habits the student should adopt (online or offline).
4. Structured output in bullet points, not letter format.
5. Suggestions should be tailored to the student's learning style, motivation, attention span, and performance.
"""

    return prompt

In [6]:
sample_student = {
    "Name": "John Doe",
    "Age": 15,
    "Gender": "Male",
    "Class_Type": "Junior Secondary",
    "Parent_Education": "Bachelors Degree",
    "School_Type": "Public",
    "Health": "Healthy",
    "Disability": "None",
    "Socioeconomic_Status": "Medium",
    "Performance": 72,
    "Attendance": 88,
    "Dropout_Risk": "Low",
    "Tuition_Paid": "Yes",
    "Teacher_Comment": "Shows potential but needs to focus more on math",
    "Test_Scores": [15, 18, 14],
    "Assignment_Completion_Rate": 90,
    "Average_GPA": 3.5,
    "Course_Grades": {"Math": 70, "English": 80, "Science": 75},
    "Course_Interest_Level": {"Math": 7, "English": 9, "Science": 8},
    "Time_Spent_Per_Subject": {"Math": 5, "English": 3, "Science": 4},
    "Course_Difficulty_Perception": {"Math": 8, "English": 5, "Science": 6},
    "Platform_Login_Frequency": 6,
    "Course_Interaction_Logs": {"Videos_Watched": "up-to-date", "Discussion_Posts": "yes", "Quiz_Attempts": 3},
    "Late_Submission_Rate": 5,
    "Learning_Style": "Visual",
    "Attention_Span": "Moderate",
    "Preferred_Learning_Pace": "Good",
    "Curiosity_Level": "High",
    "Procrastination_Level": "Low",
    "Study_Consistency": "Consistent",
    "Focus_Quality": "High",
    "Memory_Retention": "High",
    "Sleep_Habits": "Healthy",
    "Self_Regulation": "Good",
    "Goal_Setting": "Strong",
    "Personality": "Conscientiousness",
    "Motivation_Type": "Intrinsic",
    "Stress_Level": "Moderate",
    "Social_Interaction_Level": "Medium",
    "Confidence_Level": "High",
    "Resilience": "Good",
    "Growth_Mindset": "Strong",
    "Grit": "High",
    "Anxiety_Level": "Low",
    "Course_Recommendation_System_Output": ["Math", "Science"],
    "Inactivity_Rate_System_Output": 10,
    "Failure_Rate_System_Output": 5
}

In [7]:
prompt_text = generate_prompt(sample_student)
#Print(prompt_text)

In [8]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    provider="novita",
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="meta-llama/Llama-3.2-3B-Instruct",
    messages=[{"role": "user", "content": prompt_text}]
)

print(completion.choices[0].message)

ChatCompletionOutputMessage(role='assistant', content="**Personalized Feedback and Recommendations**\n\nJohn, it's great to see your consistent effort in maintaining a healthy lifestyle, good attendance, and strong motivation. Your academic performance is commendable, and your teacher's comment points to areas where you can improve. As you continue to grow, remember that learning is a lifelong journey. Focus on building upon your strengths while addressing the challenges in Math.\n\n**Recommendations for Academic Improvement and Personal Growth:**\n\n• **Develop a Math Routine:** Since your teacher mentioned that you need to focus more on Math, create a routine to help you stay on top of your assignments and homework. Allocate specific times each day or week for Math practice, and consider working with a tutor or classmate who excels in the subject.\n• **Explore Math Applications:** Engage with real-world Math problems and applications that interest you. This could be through online re

In [9]:
# Save prompt template
with open("prompt_template.txt", "w") as f:
    f.write(prompt_text)

# ✅ Correct way to save Hugging Face model config
hf_config = {
    "model": "meta-llama/Llama-3.2-3B-Instruct"
}
import json
with open("hf_client_config.json", "w") as f:
    json.dump(hf_config, f)